# <span style="color: #1F1DB5;">SPRINT 9 – Introducción al Machine Learning Supervisado

# <span style="color: #1F1DB5;">Notebook 2 – Primer Modelo y KNN — Versión estudiante

## Cómo usar esta versión estudiante

Este notebook está preparado para que tomes apuntes, completes ejercicios y documentes tus decisiones durante la clase.

**Modo de trabajo recomendado:**

1. Lee primero la consigna de cada bloque o ejercicio.
2. Completa los espacios de respuesta antes de comparar con la explicación del instructor/a.
3. Ejecuta las celdas de setup cuando existan.
4. Escribe interpretación, dudas y decisiones en los espacios de apuntes.
5. Al final, revisa si tus respuestas conectan datos, método e implicación de negocio.

> Las salidas de ejecución fueron limpiadas para que puedas correr el notebook desde cero.


## Notas de clase principales

- Comprender el flujo completo de train/test split y por qué es fundamental para evaluar modelos.
- Entender la intuición detrás del algoritmo K-Nearest Neighbors (KNN) y su analogía con "dime con quién andas...".
- Diferenciar los tipos de distancia (Euclidiana, Manhattan, Coseno) y cuándo usar cada una.
- Analizar el efecto del hiperparámetro k en las fronteras de decisión.
- Aplicar StandardScaler para normalizar features antes de usar KNN.

### Mis notas iniciales

- 
- 
- 


## <span style="color: #2826DE;">Objetivos de Aprendizaje

- Comprender el flujo completo de **train/test split** y por qué es fundamental para evaluar modelos.
- Entender la intuición detrás del algoritmo **K-Nearest Neighbors (KNN)** y su analogía con "dime con quién andas...".
- Diferenciar los tipos de **distancia** (Euclidiana, Manhattan, Coseno) y cuándo usar cada una.
- Analizar el **efecto del hiperparámetro k** en las fronteras de decisión.
- Aplicar **StandardScaler** para normalizar features antes de usar KNN.
- Construir un **primer modelo de clasificación** con sklearn y el dataset Iris.

### <span style="color: #2772F2;">Agenda (2 horas)

| Tema | Duración |
|---|---|
Flujo de entrenamiento: train/test split | 15 min |
KNN: Intuición y analogía | 15 min |
Tipos de distancia | 15 min |
Efecto de k en fronteras de decisión | 15 min |
StandardScaler e importancia del escalado | 10 min |
Actividad práctica – Breakout Rooms | 15 min |
Tips y buenas prácticas | 10 min |
Cierre y Kahoot | 5 min |

## <span style="color: #2826DE;">❓ Pregunta Guía

### Cuando no conocemos la respuesta para un caso nuevo, ¿qué tan buena estrategia es asumir que se comportará como casos parecidos que ya conocemos?

### Mi respuesta inicial

- 


## <span style="color: #2826DE;">Flujo de entrenamiento: train/test split (15 mins)

Imagina que estudias para un examen practicando solo con las respuestas frente a ti. ¿Realmente aprendiste o solo memorizaste? Ese es exactamente el problema que resolvemos con **train/test split**.

En Machine Learning, necesitamos una forma honesta de evaluar si nuestro modelo realmente aprendió patrones generalizables o si simplemente memorizó los datos. La solución es separar nuestros datos en dos conjuntos:

- **Conjunto de entrenamiento (train):** Los datos con los que el modelo aprende. Es como el material de estudio.
- **Conjunto de prueba (test):** Datos que el modelo NUNCA ha visto. Es como el examen final.

La proporción más común es **80/20** o **75/25**. Si usamos todos los datos para entrenar, no tenemos forma de saber si el modelo generaliza bien a datos nuevos — un fenómeno llamado **sobreajuste** (overfitting).

Vamos a implementar train/test split con sklearn. Observa cómo `random_state` nos permite reproducir exactamente la misma partición cada vez que ejecutamos el código:

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris

# Cargamos el dataset Iris: 150 flores, 4 features, 3 clases
iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target, name='species')

print(f"Dataset completo: {X.shape[0]} muestras, {X.shape[1]} features")
print(f"Clases: {iris.target_names}")

# Separamos 80% entrenamiento, 20% prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nEntrenamiento: {X_train.shape[0]} muestras")
print(f"Prueba: {X_test.shape[0]} muestras")
print(f"\nProporción de clases en train:\n{y_train.value_counts(normalize=True).round(3)}")
print(f"\nProporción de clases en test:\n{y_test.value_counts(normalize=True).round(3)}")

Preguntas:

- ¿Qué pasaría si no usáramos `stratify=y` en un dataset desbalanceado?

- ¿Por qué es importante fijar `random_state` al experimentar?

- ¿Qué riesgos hay si usamos un test_size demasiado pequeño (por ejemplo, 5%)?

### Mis respuestas de validación

- 
- 


## <span style="color: #2826DE;">KNN: Intuición y analogía (15 mins)

**"Dime con quién andas y te diré quién eres."** Este refrán mexicano captura perfectamente la esencia del algoritmo K-Nearest Neighbors.

KNN es uno de los algoritmos más intuitivos en Machine Learning. Su lógica es sencilla: para clasificar un nuevo punto, observa a sus **k vecinos más cercanos** y asigna la clase que sea mayoría entre ellos.

**Analogía del mundo real:** Llegas a una fiesta donde no conoces a nadie. ¿Cómo decides en qué grupo integrarte? Probablemente te acercas al grupo de personas que se parecen más a ti (mismos intereses, edad similar, forma de vestir). Eso es exactamente lo que hace KNN.

**Características clave de KNN:**
- Es un algoritmo **lazy learner** (perezoso): no aprende un modelo explícito durante el entrenamiento. Solo guarda los datos.
- En la predicción, calcula la distancia del nuevo punto a TODOS los puntos de entrenamiento.
- Es **no paramétrico**: no asume ninguna distribución en los datos.
- Su complejidad computacional crece con el tamaño del dataset (costoso para datasets grandes).

Veamos visualmente cómo KNN clasifica un punto nuevo basándose en sus vecinos más cercanos:

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

# Visualización del concepto KNN
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Datos sintéticos para ilustrar
np.random.seed(42)
clase_a = np.random.randn(15, 2) + np.array([2, 2])
clase_b = np.random.randn(15, 2) + np.array([5, 5])
nuevo_punto = np.array([3.5, 3.8])

for ax, k in zip(axes, [3, 7]):
    ax.scatter(clase_a[:, 0], clase_a[:, 1], c='blue', label='Clase A', s=60)
    ax.scatter(clase_b[:, 0], clase_b[:, 1], c='red', label='Clase B', s=60)
    ax.scatter(*nuevo_punto, c='green', marker='*', s=300, label='Nuevo punto', zorder=5)
    
    # Calcular distancias y encontrar k vecinos
    todos = np.vstack([clase_a, clase_b])
    distancias = np.sqrt(((todos - nuevo_punto)**2).sum(axis=1))
    k_idx = distancias.argsort()[:k]
    
    # Dibujar círculo que engloba k vecinos
    radio = distancias[k_idx[-1]]
    circulo = Circle(nuevo_punto, radio, fill=False, linestyle='--', color='green', linewidth=2)
    ax.add_patch(circulo)
    
    # Resaltar vecinos
    ax.scatter(todos[k_idx, 0], todos[k_idx, 1], facecolors='none', 
               edgecolors='green', s=150, linewidth=2)
    
    clase_a_count = sum(1 for i in k_idx if i < 15)
    clase_b_count = k - clase_a_count
    ax.set_title(f'k={k} → Clase {"A" if clase_a_count > clase_b_count else "B"} '
                 f'({clase_a_count}A vs {clase_b_count}B)', fontsize=12)
    ax.legend()
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

plt.suptitle('KNN: El efecto de k en la clasificación', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

Preguntas:

- ¿Qué pasa si k=1? ¿Y si k es igual al número total de muestras?

- ¿Por qué se recomienda usar valores impares para k en clasificación binaria?

- ¿En qué situaciones crees que KNN NO sería una buena elección?

### Mis respuestas de validación

- 
- 


## <span style="color: #2826DE;">Tipos de distancia (15 mins)

El concepto de "cercanía" en KNN depende de cómo medimos la distancia. No existe una sola forma correcta — la mejor métrica depende de la naturaleza de tus datos.

### Distancia Euclidiana (la línea recta)
Es la distancia "como vuela el pájaro". La más común e intuitiva.

$$d(p, q) = \sqrt{\sum_{i=1}^{n} (p_i - q_i)^2}$$

**Cuándo usarla:** Cuando todas las features tienen escalas similares y representan magnitudes continuas (altura, peso, temperatura).

### Distancia Manhattan (la cuadra)
Es la distancia recorrida si solo pudieras moverte en ángulos de 90° — como caminar por las calles de una ciudad.

$$d(p, q) = \sum_{i=1}^{n} |p_i - q_i|$$

**Cuándo usarla:** Cuando las features representan movimientos en cuadrícula o cuando quieres reducir el efecto de outliers (es más robusta que la Euclidiana).

### Distancia Coseno (el ángulo)
No mide la magnitud, sino la dirección. Dos vectores que apuntan en la misma dirección tienen distancia coseno de 0, sin importar su longitud.

$$d(p, q) = 1 - \frac{p \cdot q}{||p|| \cdot ||q||}$$

**Cuándo usarla:** En NLP y text mining, recomendaciones, o cuando la magnitud no importa, solo la proporción entre features.

Implementemos las tres distancias y comparemos sus resultados con un ejemplo concreto:

In [ ]:
from scipy.spatial.distance import euclidean, cityblock, cosine

# Ejemplo: dos estudiantes con diferentes características
estudiante_a = np.array([8, 90, 3])   # horas_estudio, calificacion, proyectos
estudiante_b = np.array([6, 75, 5])
estudiante_c = np.array([16, 180, 6])  # Mismo "perfil" que A pero escalado x2

print("Comparando distancias entre estudiantes:")
print(f"  Estudiante A: {estudiante_a} (horas, calif, proyectos)")
print(f"  Estudiante B: {estudiante_b}")
print(f"  Estudiante C: {estudiante_c} (A multiplicado por 2)\n")

print("Distancia A→B:")
print(f"  Euclidiana: {euclidean(estudiante_a, estudiante_b):.4f}")
print(f"  Manhattan:  {cityblock(estudiante_a, estudiante_b):.4f}")
print(f"  Coseno:     {cosine(estudiante_a, estudiante_b):.4f}")

print(f"\nDistancia A→C:")
print(f"  Euclidiana: {euclidean(estudiante_a, estudiante_c):.4f}")
print(f"  Manhattan:  {cityblock(estudiante_a, estudiante_c):.4f}")
print(f"  Coseno:     {cosine(estudiante_a, estudiante_c):.4f}")

print("\n→ Nota: Coseno dice que A y C son idénticos (misma dirección),")
print("  mientras Euclidiana y Manhattan los ven muy diferentes (distinta magnitud).")

Preguntas:

- ¿Por qué la distancia coseno ve a A y C como idénticos pero Euclidiana no?

- Si tus features son "número de palabras por documento", ¿qué distancia usarías y por qué?

- ¿Qué efecto tiene un outlier extremo en la distancia Euclidiana vs Manhattan?

### Mis respuestas de validación

- 
- 


## <span style="color: #2826DE;">Efecto de k en fronteras de decisión (15 mins)

El hiperparámetro **k** controla la complejidad del modelo KNN:

- **k pequeño (ej. k=1):** El modelo es muy sensible a puntos individuales. Captura todos los detalles (y el ruido). Fronteras de decisión irregulares y complejas. → **Alto riesgo de overfitting.**
- **k grande (ej. k=50):** El modelo suaviza las fronteras de decisión. Ignora patrones locales. → **Alto riesgo de underfitting.**

La clave es encontrar el **balance**: un k que capture los patrones reales sin ajustarse al ruido.

**Regla práctica:** Comienza con k = √n (raíz cuadrada del número de muestras) y ajusta a partir de ahí.

Visualicemos cómo cambian las fronteras de decisión con diferentes valores de k en el dataset Iris (usando solo 2 features para poder graficar):

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from matplotlib.colors import ListedColormap

# Usamos solo 2 features para visualizar en 2D
X_vis = X_train.iloc[:, :2].values
y_vis = y_train.values

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
k_values = [1, 5, 25]
cmap_light = ListedColormap(['#FFAAAA', '#AAFFAA', '#AAAAFF'])
cmap_bold = ListedColormap(['#FF0000', '#00FF00', '#0000FF'])

for ax, k in zip(axes, k_values):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_vis, y_vis)
    
    # Crear malla para visualizar fronteras
    h = 0.02
    x_min, x_max = X_vis[:, 0].min() - 1, X_vis[:, 0].max() + 1
    y_min, y_max = X_vis[:, 1].min() - 1, X_vis[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    
    Z = knn.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, cmap=cmap_light, alpha=0.6)
    ax.scatter(X_vis[:, 0], X_vis[:, 1], c=y_vis, cmap=cmap_bold, 
               edgecolors='k', s=30)
    
    accuracy = knn.score(X_test.iloc[:, :2].values, y_test.values)
    ax.set_title(f'k = {k} (accuracy: {accuracy:.2f})')
    ax.set_xlabel(iris.feature_names[0])
    ax.set_ylabel(iris.feature_names[1])

plt.suptitle('Fronteras de decisión de KNN con diferentes valores de k', fontweight='bold')
plt.tight_layout()
plt.show()

## <span style="color: #2826DE;">StandardScaler e importancia del escalado (10 mins)

¿Recuerdas que KNN depende de la distancia? Esto genera un problema grave cuando las features tienen escalas muy diferentes.

**Ejemplo:** Si una feature es "edad" (rango 0-100) y otra es "ingreso anual" (rango 0-1,000,000), la distancia estará completamente dominada por el ingreso. ¡La edad se vuelve irrelevante!

**StandardScaler** resuelve esto transformando cada feature para que tenga:
- Media = 0
- Desviación estándar = 1

La fórmula es: $z = \frac{x - \mu}{\sigma}$

**Regla de oro:** Si tu algoritmo usa distancias (KNN, SVM) o gradientes (redes neuronales), **SIEMPRE** escala tus features.

Comparemos el accuracy de KNN con y sin escalado para demostrar el impacto:

In [ ]:
from sklearn.preprocessing import StandardScaler

# Sin escalado
knn_sin_escalar = KNeighborsClassifier(n_neighbors=5)
knn_sin_escalar.fit(X_train, y_train)
acc_sin_escalar = knn_sin_escalar.score(X_test, y_test)

# Con escalado
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # ¡IMPORTANTE! Solo transform en test

knn_con_escalar = KNeighborsClassifier(n_neighbors=5)
knn_con_escalar.fit(X_train_scaled, y_train)
acc_con_escalar = knn_con_escalar.score(X_test_scaled, y_test)

print(f"Accuracy SIN escalar: {acc_sin_escalar:.4f}")
print(f"Accuracy CON escalar: {acc_con_escalar:.4f}")
print(f"\n⚠️  IMPORTANTE: fit_transform() en TRAIN, solo transform() en TEST")
print("   Si haces fit en test, introduces data leakage.")

# Visualizar el efecto del escalado
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].boxplot(X_train.values, labels=[f.split(' (')[0] for f in X_train.columns])
axes[0].set_title('Features SIN escalar')
axes[0].tick_params(axis='x', rotation=45)

axes[1].boxplot(X_train_scaled, labels=[f.split(' (')[0] for f in X_train.columns])
axes[1].set_title('Features CON StandardScaler')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

Preguntas:

- ¿Qué es "data leakage" y por qué hacer `fit` en el test set lo provoca?

- ¿Los árboles de decisión necesitan escalado? ¿Por qué sí o por qué no?

- ¿Qué pasa si una feature tiene muchos outliers — StandardScaler sigue siendo la mejor opción?

### Mis respuestas de validación

- 
- 


## <span style="color: #2826DE;">Construyendo nuestro primer modelo completo (10 mins)

Ya tenemos todas las piezas: sabemos dividir datos, entendemos KNN, conocemos las distancias y la importancia del escalado. Ahora armemos el pipeline completo de clasificación paso a paso.

El flujo profesional para entrenar y evaluar un modelo de clasificación es:

1. **Cargar y explorar** los datos.
2. **Dividir** en train/test con stratify.
3. **Escalar** features (fit en train, transform en test).
4. **Entrenar** el modelo.
5. **Evaluar** en el test set.
6. **Experimentar** con hiperparámetros.

Implementemos el flujo completo y busquemos el mejor k evaluando accuracy para cada valor:

In [ ]:
from sklearn.metrics import classification_report

# Flujo completo profesional
# Paso 1-2: Ya tenemos X_train, X_test, y_train, y_test
# Paso 3: Escalado
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Paso 4-5: Entrenar y evaluar con diferentes k
k_range = range(1, 31)
scores = []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_s, y_train)
    scores.append(knn.score(X_test_s, y_test))

# Encontrar el mejor k
best_k = k_range[np.argmax(scores)]
print(f"Mejor k: {best_k} con accuracy: {max(scores):.4f}")

# Modelo final con el mejor k
knn_final = KNeighborsClassifier(n_neighbors=best_k)
knn_final.fit(X_train_s, y_train)
y_pred = knn_final.predict(X_test_s)

print(f"\nReporte de clasificación (k={best_k}):")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

# Visualización de accuracy vs k
plt.figure(figsize=(10, 4))
plt.plot(k_range, scores, 'bo-', markersize=4)
plt.axvline(best_k, color='red', linestyle='--', label=f'Mejor k={best_k}')
plt.xlabel('Valor de k')
plt.ylabel('Accuracy en test set')
plt.title('Búsqueda del mejor k para KNN')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Preguntas:

- ¿La curva de accuracy vs k tiene forma de U invertida? ¿Por qué?

- ¿Evaluar muchos valores de k en el test set es una buena práctica o genera sesgo?

- ¿Cómo podrías usar cross-validation para elegir k de forma más robusta?

### Mis respuestas de validación

- 
- 


## <span style="color: #2826DE;">Actividad práctica – Breakout Rooms (15 mins)

### Espacio de trabajo del estudiante

**Respuesta, decisiones o interpretación:**

- 
- 


### <span style="color: #aa0808;">En esta ocasión, quien compartirá resultados será: (Giremos la ruleta)

### Instrucciones:

**Objetivo:** Construir un clasificador KNN completo y experimentar con hiperparámetros.

1. Carga el dataset Iris con `load_iris()`.
2. Divide en train/test con 75/25.
3. Aplica `StandardScaler` correctamente (fit en train, transform en test).
4. Entrena KNN con k=3, k=7, k=15 y compara accuracy.
5. Grafica accuracy vs k para k de 1 a 30.
6. Responde: ¿cuál es el mejor k? ¿La curva muestra overfitting/underfitting?

**Bonus:** Prueba con `metric='manhattan'` y compara resultados.

**Tiempo:** 12 minutos de trabajo + 3 minutos de discusión.

## <span style="color: #2826DE;">Tips y buenas prácticas

- **Siempre escala** antes de KNN. Es el error #1 de principiantes.
- Usa `stratify=y` en `train_test_split` cuando tus clases están desbalanceadas.
- KNN no funciona bien con **muchas dimensiones** (maldición de la dimensionalidad) — considera reducir features primero.
- Para datasets grandes, considera `BallTree` o `KDTree` como algoritmo interno (parámetro `algorithm` en sklearn).
- El valor de k ideal depende del dataset — siempre experimentar.
- Recuerda que KNN es **computacionalmente costoso** en predicción: debe calcular distancias contra TODOS los puntos de entrenamiento.

## <span style="color: #2826DE;">Cierre y Kahoot (5 mins)

Kahoot - Preguntas sugeridas

1️⃣ ¿Qué tipo de algoritmo es KNN?

- Paramétrico y eager learner
- No paramétrico y lazy learner 
- Paramétrico y lazy learner
- No paramétrico y eager learner


2️⃣ ¿Qué distancia mide el ángulo entre vectores en lugar de su magnitud?

- Euclidiana
- Manhattan
- Coseno 
- Chebyshev


3️⃣ ¿Qué pasa si usamos k=1 en KNN?

- El modelo se vuelve demasiado simple
- El modelo memoriza el ruido de los datos 
- El modelo no puede hacer predicciones
- El modelo siempre predice la clase mayoritaria


4️⃣ ¿Por qué es importante usar StandardScaler con KNN?

- Porque KNN requiere datos categóricos
- Porque las features con mayor escala dominan el cálculo de distancias 
- Porque reduce el número de features
- Porque elimina los outliers automáticamente


5️⃣ Al hacer train/test split, ¿cuándo se debe usar fit_transform?

- Solo en el conjunto de prueba
- En ambos conjuntos juntos
- Solo en el conjunto de entrenamiento 
- No importa el orden


6️⃣ ¿Cuál es una desventaja principal de KNN en datasets grandes?

- No puede manejar más de 2 clases
- Es computacionalmente costoso en predicción 
- Requiere que los datos sean linealmente separables
- No funciona con features numéricas